# 06. Shortest Paths: Dijkstra, Bellman-Ford, Floyd-Warshall

When edges have **weights**, BFS is no longer enough. Which algorithm depends on the graph.

**Learning order:** CHOOSING → DIJKSTRA → NETWORK DELAY → MIN EFFORT PATH → BELLMAN-FORD → CHEAPEST FLIGHTS → 0-1 BFS → FLOYD-WARSHALL → INTERVIEW THINKING

## 1. Which algorithm?

| Situation | Algorithm | Time |
|---|---|---|
| unweighted | BFS | O(V + E) |
| non-negative weights, one source | **Dijkstra** (heap) | O((V + E) log V) |
| negative weights, or "at most K edges" | **Bellman-Ford** | O(V · E) |
| all pairs, small V (≤ ~400) | **Floyd-Warshall** | O(V³) |
| weights are only 0 and 1 | 0-1 BFS (deque) | O(V + E) |
| DAG with weights | topological order + relax | O(V + E) |

## 2. Dijkstra's algorithm

**Idea:** always settle the **closest unsettled vertex** (min-heap). Once a vertex is popped with distance `d`, `d` is final **because all edges are non-negative** (a longer detour can only add more).

1. `dist[src] = 0`, push `(0, src)`.
2. Pop `(d, u)`. If `d > dist[u]`, skip it (stale entry).
3. For each `(v, w)`: if `d + w < dist[v]`, update and push.

Real life: Google Maps fastest route.

In [1]:
import heapq
from collections import defaultdict, deque

def dijkstra(graph, src):
    dist = {src: 0}
    heap = [(0, src)]
    while heap:
        d, u = heapq.heappop(heap)
        if d > dist.get(u, float("inf")):               # stale heap entry
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist.get(v, float("inf")):
                dist[v] = nd
                heapq.heappush(heap, (nd, v))
    return dist

g = {"A": [("B", 1), ("C", 4)], "B": [("C", 2), ("D", 5)], "C": [("D", 1)], "D": []}
print(dijkstra(g, "A"))   # {'A': 0, 'B': 1, 'C': 3, 'D': 4}

{'A': 0, 'B': 1, 'C': 3, 'D': 4}


**Why Dijkstra fails with negative edges:** a vertex may be settled with distance 5, but a later path through a `-10` edge could reach it with `-3`. The "final" claim is false.

## 3. Network Delay Time

A signal starts at `k`. Time for **all** nodes to receive it = the **maximum** shortest distance. If some node is unreachable → `-1`.

In [2]:
def network_delay_time(times, n, k):
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    dist = dijkstra(graph, k)
    return max(dist.values()) if len(dist) == n else -1

print(network_delay_time([[2, 1, 1], [2, 3, 1], [3, 4, 1]], 4, 2), network_delay_time([[1, 2, 1]], 2, 2))   # 2 -1

2 -1


## 4. Path With Minimum Effort (Dijkstra on a grid)

Effort of a path = the **maximum absolute height difference** between consecutive cells. Minimize it. Same Dijkstra, but the "distance" combines with `max` instead of `+`: `new_effort = max(current_effort, diff)`.

In [3]:
def minimum_effort_path(heights):
    rows, cols = len(heights), len(heights[0])
    best = [[float("inf")] * cols for _ in range(rows)]
    best[0][0] = 0
    heap = [(0, 0, 0)]
    while heap:
        effort, r, c = heapq.heappop(heap)
        if (r, c) == (rows - 1, cols - 1):
            return effort
        if effort > best[r][c]:
            continue
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                ne = max(effort, abs(heights[nr][nc] - heights[r][c]))
                if ne < best[nr][nc]:
                    best[nr][nc] = ne
                    heapq.heappush(heap, (ne, nr, nc))

print(minimum_effort_path([[1, 2, 2], [3, 8, 2], [5, 3, 5]]), minimum_effort_path([[1, 2, 3], [3, 8, 4], [5, 3, 5]]))   # 2 1

2 1


## 5. Bellman-Ford and "at most K edges"

**Idea:** relax **all edges**, repeated `V - 1` times. After round `i`, distances using **at most `i` edges** are correct. It handles **negative edges** (and detects negative cycles with one more round).

For "at most `k` stops" problems, run only `k + 1` rounds, and **use a copy** of the previous round's distances, so each round extends paths by exactly one edge.

In [4]:
def find_cheapest_price(n, flights, src, dst, k):
    prices = [float("inf")] * n
    prices[src] = 0
    for _ in range(k + 1):                          # k stops = k + 1 edges
        nxt = prices[:]                             # copy: do not chain edges within one round
        for u, v, w in flights:
            if prices[u] + w < nxt[v]:
                nxt[v] = prices[u] + w
        prices = nxt
    return -1 if prices[dst] == float("inf") else prices[dst]

flights = [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]]
print(find_cheapest_price(4, flights, 0, 3, 1))                                         # 700
print(find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1))        # 200
print(find_cheapest_price(3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0))        # 500

700
200
500


## 6. 0-1 BFS

Edge weights are `0` or `1`. Use a **deque**: a `0` edge pushes the neighbour to the **front** (same distance), a `1` edge to the **back**. No heap needed.

In [5]:
def zero_one_bfs(n, edges, src):                    # edges: (u, v, w) with w in {0, 1}, directed
    graph = defaultdict(list)
    for u, v, w in edges:
        graph[u].append((v, w))
    dist = [float("inf")] * n
    dist[src] = 0
    dq = deque([src])
    while dq:
        u = dq.popleft()
        for v, w in graph[u]:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                dq.appendleft(v) if w == 0 else dq.append(v)
    return dist

print(zero_one_bfs(4, [(0, 1, 0), (1, 2, 1), (0, 2, 1), (2, 3, 0)], 0))   # [0, 0, 1, 1]

[0, 0, 1, 1]


## 7. Floyd-Warshall (all pairs)

`dist[i][j]` = shortest path from `i` to `j`. For every **intermediate** vertex `k`, check if going `i → k → j` is shorter. Three nested loops, **`k` must be the outer loop**.

Use when V is small and you need distances between **all** pairs.

In [6]:
def floyd_warshall(n, edges):
    INF = float("inf")
    dist = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for u, v, w in edges:
        dist[u][v] = min(dist[u][v], w)
    for k in range(n):
        for i in range(n):
            for j in range(n):
                if dist[i][k] + dist[k][j] < dist[i][j]:
                    dist[i][j] = dist[i][k] + dist[k][j]
    return dist

edges = [(0, 1, 3), (0, 3, 7), (1, 0, 8), (1, 2, 2), (2, 0, 5), (2, 3, 1), (3, 0, 2)]
print(floyd_warshall(4, edges))   # [[0, 3, 5, 6], [5, 0, 2, 3], [3, 6, 0, 1], [2, 5, 7, 0]]

[[0, 3, 5, 6], [5, 0, 2, 3], [3, 6, 0, 1], [2, 5, 7, 0]]


## 8. Interview thinking

**Signals:** "cheapest / fastest route", "minimum cost to reach", "at most K stops", "minimum of the maximum", "time for a signal to reach everyone".

**Say:** "Weights are non-negative, so Dijkstra with a min-heap: O((V + E) log V)." / "At most K edges, so Bellman-Ford limited to K + 1 rounds."

**Common mistakes:** BFS on weighted graphs, Dijkstra with negative edges, no stale-entry check (still correct but slower), forgetting the copy in the K-stops Bellman-Ford (paths grow too long), wrong loop order in Floyd-Warshall, `INF` overflow when adding (use `float("inf")`).

## Quick Revision

- Unweighted: BFS. Non-negative: Dijkstra. Negative / K edges: Bellman-Ford. All pairs: Floyd. 0/1: deque BFS.
- Dijkstra: heap, skip stale entries, relax neighbours.
- Grid Dijkstra: change `+` to `max` for "minimum effort".
- K stops: `k + 1` rounds with a copy of the previous distances.